# 🎯 DSI EDA Template — Squarepoint Capital Onsite Prep
## Dataset: `[INSERT DATASET NAME]` | Date: `[DATE]` | Timer: 3.5h

---

### Onsite mindset
> The interviewer is not evaluating your model accuracy.  
> They are evaluating **how you think** under pressure.  
> Narrate every decision. Say why, not just what.

### Session structure
| Phase | Time | Goal |
|-------|------|------|
| **0. Setup & first look** | 0:00–0:10 | Load data, understand shape, spot problems |
| **1. Data quality** | 0:10–0:30 | Missing, dtypes, duplicates, leakage check |
| **2. Univariate EDA** | 0:30–0:55 | Distributions, outliers, target variable |
| **3. Bivariate EDA** | 0:55–1:25 | Correlations, group differences, time patterns |
| **4. Hypothesis formation** | 1:25–1:40 | Write 3 hypotheses explicitly before modelling |
| **5. Feature engineering** | 1:40–2:10 | Build 3–5 features, test IC |
| **6. Model + validation** | 2:10–2:50 | TimeSeriesSplit or purged CV, no shuffling |
| **7. Findings summary** | 2:50–3:20 | What did you find? What would you do next? |

---


## ⏱️ Phase 0: Setup & First Look `[0:00–0:10]`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Timer start ────────────────────────────────────────────────────────────
from datetime import datetime
SESSION_START = datetime.now()
def elapsed():
    delta = datetime.now() - SESSION_START
    mins  = int(delta.total_seconds() // 60)
    secs  = int(delta.total_seconds() % 60)
    return f"[{mins:02d}:{secs:02d}]"

print(f"Session started: {SESSION_START.strftime('%H:%M:%S')}")

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 110, 'axes.facecolor': '#0d1117', 'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e', 'text.color': '#c9d1d9',
    'grid.color': '#21262d', 'grid.alpha': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
})
BLUE='#388bfd'; GREEN='#3fb950'; RED='#f85149'; AMBER='#f7931a'; GRAY='#8b949e'

# ── CHANGE THIS PATH ───────────────────────────────────────────────────────
PATH = '/kaggle/input/YOUR-DATASET-SLUG/'

# ── Load data ──────────────────────────────────────────────────────────────
# df = pd.read_csv(PATH + 'train.csv')
# Uncomment and modify as needed:
# df_test  = pd.read_csv(PATH + 'test.csv')
# df_meta  = pd.read_csv(PATH + 'metadata.csv')

# ── First look — do this BEFORE anything else ──────────────────────────────
# df.head(10)


In [ ]:
# ── Structural snapshot — run this cell immediately ───────────────────────
# Replace 'df' with your dataframe name

def quick_overview(df, name='df'):
    print(f"{'='*55}")
    print(f"DATASET: {name}")
    print(f"{'='*55}")
    print(f"Shape:        {df.shape[0]:,} rows  x  {df.shape[1]} columns")
    print(f"Memory:       {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
    print(f"Duplicates:   {df.duplicated().sum():,} rows")
    print()

    # Column types
    type_summary = df.dtypes.value_counts()
    print("Column types:")
    for dtype, count in type_summary.items():
        print(f"  {str(dtype):15s}: {count}")
    print()

    # Missing values
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing):
        print(f"Missing values ({len(missing)} columns):")
        for col, n in missing.head(10).items():
            print(f"  {col:30s}: {n:>6,}  ({n/len(df):.1%})")
        if len(missing) > 10: print(f"  ... and {len(missing)-10} more")
    else:
        print("Missing values: NONE ✅")
    print()

    # Target variable (last column heuristic)
    last_col = df.columns[-1]
    print(f"Last column (potential target): '{last_col}'")
    if df[last_col].dtype in ['int64','float64']:
        print(f"  dtype: {df[last_col].dtype} | mean: {df[last_col].mean():.4f} | "
              f"std: {df[last_col].std():.4f}")
        if df[last_col].nunique() <= 10:
            print(f"  value counts: {df[last_col].value_counts().to_dict()}")
    print(f"{'='*55}")
    return missing

# quick_overview(df)
print(f"{elapsed()} Ready for overview")


---
## 🔍 Phase 1: Data Quality `[0:10–0:30]`

**Before any EDA, always check:**
- [ ] Temporal leakage — are future values visible in features?
- [ ] Label leakage — does any feature encode the target directly?
- [ ] Duplicates in time series — same timestamp twice?
- [ ] Data type mismatches — dates stored as strings?
- [ ] Target distribution — is it skewed, bimodal, has outliers?


In [ ]:
# ── Leakage check framework ───────────────────────────────────────────────
# CRITICAL for financial data — run before touching features

def check_leakage_signals(df, target_col=None):
    """Print correlation of all numeric cols with target.
    High correlation (>0.9) of a feature with target is a red flag."""
    if target_col is None:
        print("Specify target_col to run leakage check")
        return

    numerics = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in numerics: numerics.remove(target_col)

    corrs = {}
    for col in numerics:
        valid = df[[col, target_col]].dropna()
        if len(valid) > 50:
            r, _ = stats.spearmanr(valid[col], valid[target_col])
            corrs[col] = abs(r)

    corr_series = pd.Series(corrs).sort_values(ascending=False)
    print(f"Top correlations with '{target_col}':")
    print(corr_series.head(15).round(4).to_string())

    suspects = corr_series[corr_series > 0.90]
    if len(suspects):
        print(f"\n⚠️  LEAKAGE SUSPECTS (r > 0.90): {list(suspects.index)}")
    else:
        print("\n✅ No obvious leakage suspects")
    return corr_series

# check_leakage_signals(df, target_col='target')
print(f"{elapsed()} Leakage check ready")


In [ ]:
# ── Time series integrity check ───────────────────────────────────────────
# Use if dataset has a date/time column

def check_time_integrity(df, date_col, id_col=None):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)

    print(f"Date range:    {df[date_col].min().date()} → {df[date_col].max().date()}")
    print(f"Total periods: {df[date_col].nunique()}")

    # Check for gaps
    if df[date_col].dtype == 'datetime64[ns]':
        date_diffs = df[date_col].diff().dropna()
        common_freq = date_diffs.mode()[0]
        gaps = date_diffs[date_diffs > common_freq * 2]
        if len(gaps):
            print(f"\n⚠️  {len(gaps)} time gaps detected (most common freq: {common_freq})")
            print(gaps.nlargest(5).to_string())
        else:
            print(f"\n✅ No time gaps (freq: {common_freq})")

    if id_col:
        print(f"\nUnique IDs: {df[id_col].nunique():,}")
        obs_per_id = df.groupby(id_col).size()
        print(f"Obs per ID:  min={obs_per_id.min()} | mean={obs_per_id.mean():.1f} | max={obs_per_id.max()}")

# check_time_integrity(df, date_col='date', id_col='asset_id')
print(f"{elapsed()} Time integrity check ready")


---
## 📊 Phase 2: Univariate EDA `[0:30–0:55]`

**Key questions:**
- What is the target distribution? Normal? Fat-tailed? Bimodal?
- Are there outliers that need treatment or are they signal?
- Do any features have near-zero variance (useless)?


In [ ]:
# ── Target variable deep-dive ─────────────────────────────────────────────
def plot_target(df, target_col, bins=60):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Distribution
    ax = axes[0]
    df[target_col].hist(ax=ax, bins=bins, color=BLUE, alpha=0.8, edgecolor='none')
    ax.axvline(df[target_col].mean(),   color=RED,   lw=1.5, linestyle='--', label='Mean')
    ax.axvline(df[target_col].median(), color=AMBER, lw=1.5, linestyle='--', label='Median')
    ax.set_title(f'Target: {target_col}', fontsize=11)
    ax.legend(fontsize=8)

    # QQ plot (normality check)
    ax = axes[1]
    stats.probplot(df[target_col].dropna(), plot=ax)
    ax.get_lines()[0].set(color=BLUE, alpha=0.5, markersize=2)
    ax.get_lines()[1].set(color=RED, linewidth=1.5)
    ax.set_title('Q-Q Plot (normality check)', fontsize=11)

    # Rolling stats (if time index available)
    ax = axes[2]
    rolling_mean = df[target_col].rolling(50, min_periods=10).mean()
    rolling_std  = df[target_col].rolling(50, min_periods=10).std()
    ax.plot(rolling_mean.values, color=BLUE, lw=1.5, label='Rolling mean (50)')
    ax.fill_between(range(len(rolling_mean)),
                    (rolling_mean - rolling_std).values,
                    (rolling_mean + rolling_std).values,
                    alpha=0.2, color=BLUE)
    ax.axhline(0, color=GRAY, lw=0.8, linestyle='--')
    ax.set_title('Rolling Mean ± 1 SD', fontsize=11)
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    # Stats summary
    t = df[target_col].dropna()
    print(f"n={len(t):,} | mean={t.mean():.4f} | std={t.std():.4f} | "
          f"skew={t.skew():.3f} | kurt={t.kurtosis():.3f}")
    print(f"min={t.min():.4f} | p5={t.quantile(.05):.4f} | "
          f"p95={t.quantile(.95):.4f} | max={t.max():.4f}")

    # Is it fat-tailed?
    excess_kurt = t.kurtosis()
    if excess_kurt > 3:
        print(f"\n⚠️  Fat tails detected (excess kurtosis={excess_kurt:.1f}). "
              f"Consider rank-normalisation before modelling.")
    else:
        print(f"\n✅ Approximately normal tails (excess kurtosis={excess_kurt:.1f})")

# plot_target(df, 'target')
print(f"{elapsed()} Target plot ready")


In [ ]:
# ── Feature variance & near-zero variance filter ──────────────────────────
def feature_quality_report(df, target_col=None):
    numerics = df.select_dtypes(include=[np.number]).copy()
    if target_col and target_col in numerics:
        numerics = numerics.drop(columns=[target_col])

    report = pd.DataFrame({
        'missing_pct':  (numerics.isnull().mean() * 100).round(2),
        'std':           numerics.std().round(4),
        'nunique':       numerics.nunique(),
        'skewness':      numerics.skew().round(3),
        'kurtosis':      numerics.kurtosis().round(3),
    })
    report['near_zero_var'] = report['std'] < 1e-6
    report['high_missing']  = report['missing_pct'] > 30

    print(f"Total numeric features: {len(report)}")
    print(f"Near-zero variance:     {report['near_zero_var'].sum()}")
    print(f"High missing (>30%):    {report['high_missing'].sum()}")
    print(f"High skewness (|sk|>3): {(report['skewness'].abs()>3).sum()}")

    return report.sort_values('missing_pct', ascending=False)

# feat_report = feature_quality_report(df, target_col='target')
# feat_report.head(20)
print(f"{elapsed()} Feature quality report ready")


---
## 🔗 Phase 3: Bivariate EDA `[0:55–1:25]`

**Key questions:**
- Which features correlate with the target?
- Are correlations stable over time or regime-dependent?
- Are there feature interactions?

**For financial data specifically:**
- IC (Information Coefficient) = Spearman correlation of feature with forward return
- IC > 0.05 is practically significant in most markets
- IC stability over time matters more than IC level


In [ ]:
# ── IC analysis — core quant EDA tool ─────────────────────────────────────
def compute_ic(df, features, target_col, group_col=None):
    """
    Compute Spearman IC between each feature and target.
    If group_col provided (e.g. 'era', 'date'), compute IC per group
    then report mean IC and ICIR (IC / std(IC)).
    """
    results = []

    if group_col is None:
        # Single IC across full dataset
        for feat in features:
            valid = df[[feat, target_col]].dropna()
            if len(valid) < 30: continue
            ic, pval = stats.spearmanr(valid[feat], valid[target_col])
            results.append({'feature': feat, 'IC': ic, 'p_value': pval, 'n': len(valid)})
        return pd.DataFrame(results).set_index('feature').sort_values('IC', key=abs, ascending=False)

    else:
        # Per-group IC (the right way for time series)
        ic_by_group = {}
        for feat in features:
            group_ics = []
            for grp, grp_df in df.groupby(group_col):
                valid = grp_df[[feat, target_col]].dropna()
                if len(valid) < 10: continue
                ic, _ = stats.spearmanr(valid[feat], valid[target_col])
                group_ics.append(ic)
            if group_ics:
                ic_arr = np.array(group_ics)
                ic_by_group[feat] = {
                    'mean_IC':   ic_arr.mean(),
                    'std_IC':    ic_arr.std(),
                    'ICIR':      ic_arr.mean() / (ic_arr.std() + 1e-9),
                    'IC_positive_pct': (ic_arr > 0).mean(),
                    'n_groups':  len(ic_arr),
                }
        return pd.DataFrame(ic_by_group).T.sort_values('mean_IC', key=abs, ascending=False)

# features = [c for c in df.columns if c not in ['target','date','id']]
# ic_table = compute_ic(df, features, target_col='target', group_col='era')
# ic_table.head(20).round(4)
print(f"{elapsed()} IC analysis ready")


In [ ]:
# ── IC decay curve ────────────────────────────────────────────────────────
# How long does the signal persist? Key for understanding alpha horizon.

def ic_decay(df, feature, target_col, date_col, max_lag=10):
    """
    Compute IC at lag 1, 2, ..., max_lag periods.
    Fast decay = short-horizon signal.
    Slow decay = longer-horizon, more robust signal.
    """
    df = df.sort_values(date_col).copy()
    lags, ics = [], []
    for lag in range(1, max_lag+1):
        df[f'target_fwd_{lag}'] = df[target_col].shift(-lag)
        valid = df[[feature, f'target_fwd_{lag}']].dropna()
        if len(valid) < 50: break
        ic, _ = stats.spearmanr(valid[feature], valid[f'target_fwd_{lag}'])
        lags.append(lag); ics.append(ic)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(lags, ics, color=[GREEN if v>0 else RED for v in ics], alpha=0.8)
    ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
    ax.set_title(f'IC Decay: {feature} → {target_col}', fontsize=11)
    ax.set_xlabel('Forward lag (periods)'); ax.set_ylabel('Spearman IC')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()
    return pd.DataFrame({'lag': lags, 'IC': ics})

# ic_decay(df, feature='your_feature', target_col='target', date_col='date', max_lag=10)
print(f"{elapsed()} IC decay ready")


In [ ]:
# ── Correlation heatmap (top features) ───────────────────────────────────
def plot_correlation_heatmap(df, target_col, top_n=20):
    numerics = df.select_dtypes(include=[np.number])

    # Select top N most correlated with target
    target_corrs = numerics.corrwith(numerics[target_col]).abs()
    top_features = target_corrs.nlargest(top_n+1).index.tolist()
    if target_col in top_features:
        top_features.remove(target_col)
    top_features = top_features[:top_n]

    corr_matrix = numerics[top_features].corr()

    fig, ax = plt.subplots(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, cmap='RdYlGn', center=0, annot=False,
                ax=ax, linewidths=0.2, cbar_kws={'label':'Pearson r'})
    ax.set_title(f'Feature Correlation Heatmap (top {top_n} by IC with target)', fontsize=11)
    plt.tight_layout(); plt.show()

    # Flag multicollinear pairs
    flat = corr_matrix.abs().where(~mask).stack()
    high_corr = flat[flat > 0.85]
    if len(high_corr):
        print(f"⚠️  Multicollinear pairs (|r| > 0.85):")
        print(high_corr.sort_values(ascending=False).head(10).round(3).to_string())

# plot_correlation_heatmap(df, target_col='target', top_n=20)
print(f"{elapsed()} Correlation heatmap ready")


---
## 💡 Phase 4: Hypothesis Formation `[1:25–1:40]`

**STOP. Write your hypotheses BEFORE building features.**

This is what separates quant researchers from data miners.  
A hypothesis is a testable statement about why a feature should predict the target.

---

### Template — fill this in:

```
HYPOTHESIS 1:
  Observation: [what you saw in EDA]
  Mechanism:   [why this should predict the target]
  Feature:     [how to operationalise it]
  Expected IC: [positive / negative / unknown]
  Falsification: [what would prove this wrong]

HYPOTHESIS 2:
  ...

HYPOTHESIS 3:
  ...
```

---

### Example (for a returns prediction task):

```
HYPOTHESIS 1:
  Observation:  High velocity (many transactions in 1h) correlates with price impact
  Mechanism:    Informed traders tend to split orders → velocity is a toxicity proxy
  Feature:      velocity_1h / rolling_avg_velocity_30d (normalised)
  Expected IC:  Negative (high velocity → mean reversion)
  Falsification: If IC is positive, velocity is momentum not toxicity

HYPOTHESIS 2:
  Observation:  Bid-ask spread widens before large price moves
  Mechanism:    Market makers widen spread when uncertain → spread = uncertainty signal
  Feature:      spread / rolling_avg_spread_5d
  Expected IC:  Negative for subsequent 1h return (high spread → lower return)
  Falsification: If IC near zero, spread has no directional information
```


In [ ]:
# ── Write your 3 hypotheses here ─────────────────────────────────────────
HYPOTHESES = {
    1: {
        'observation':    '',   # What did you see in EDA?
        'mechanism':      '',   # Why should this predict the target?
        'feature':        '',   # How to operationalise?
        'expected_sign':  '',   # Positive / Negative / Unknown
        'falsification':  '',   # What would prove this wrong?
    },
    2: {
        'observation':    '',
        'mechanism':      '',
        'feature':        '',
        'expected_sign':  '',
        'falsification':  '',
    },
    3: {
        'observation':    '',
        'mechanism':      '',
        'feature':        '',
        'expected_sign':  '',
        'falsification':  '',
    },
}

print(f"{elapsed()} Hypotheses documented")
for i, h in HYPOTHESES.items():
    print(f"\nHypothesis {i}:")
    for k,v in h.items(): print(f"  {k:15s}: {v or '(empty)'}")


---
## ⚙️ Phase 5: Feature Engineering `[1:40–2:10]`

**Rules for DSI:**
- Build features from your hypotheses — not randomly
- Test IC immediately after building each feature
- Keep only features with |IC| > 0.02 (anything below is noise)
- Apply rank normalisation for fat-tailed distributions


In [ ]:
# ── Feature engineering utilities ────────────────────────────────────────

def rank_normalise(series):
    """Rank-normalise to [-0.5, 0.5]. Standard in quant research."""
    return series.rank(pct=True) - 0.5

def rolling_zscore(series, window=20):
    """Z-score relative to rolling window. Removes non-stationarity."""
    roll = series.rolling(window, min_periods=window//2)
    return (series - roll.mean()) / (roll.std() + 1e-9)

def momentum(series, lookback):
    """Simple price momentum = return over lookback periods."""
    return series.pct_change(lookback)

def mean_reversion(series, window):
    """Distance from rolling mean. Negative = below mean (buy signal)."""
    roll_mean = series.rolling(window).mean()
    roll_std  = series.rolling(window).std()
    return -(series - roll_mean) / (roll_std + 1e-9)

def vol_adjusted_return(ret_series, vol_window=20):
    """Volatility-adjusted return = return / rolling vol. Vol targeting."""
    roll_vol = ret_series.rolling(vol_window).std()
    return ret_series / (roll_vol + 1e-9)

# ── Build your features here ──────────────────────────────────────────────
# df['feat_1'] = rank_normalise(df['your_raw_feature'])
# df['feat_2'] = rolling_zscore(df['price'], window=20)
# df['feat_3'] = momentum(df['price'], lookback=5)

print(f"{elapsed()} Feature engineering utils loaded")
print("Available functions: rank_normalise, rolling_zscore, momentum, mean_reversion, vol_adjusted_return")


In [ ]:
# ── Test IC of each new feature immediately ───────────────────────────────
def test_feature_ic(df, feature_col, target_col, group_col=None, verbose=True):
    """Quick IC test for a single feature. Run after building each feature."""
    if group_col:
        ics = []
        for _, grp in df.groupby(group_col):
            valid = grp[[feature_col, target_col]].dropna()
            if len(valid) < 10: continue
            ic, _ = stats.spearmanr(valid[feature_col], valid[target_col])
            ics.append(ic)
        if not ics: return None
        mean_ic = np.mean(ics)
        icir    = mean_ic / (np.std(ics) + 1e-9)
        if verbose:
            signal = '✅ KEEP' if abs(mean_ic) > 0.02 else '❌ WEAK'
            print(f"{signal} | {feature_col:30s} | mean IC={mean_ic:+.4f} | ICIR={icir:+.3f} | n_periods={len(ics)}")
        return mean_ic, icir
    else:
        valid = df[[feature_col, target_col]].dropna()
        ic, pval = stats.spearmanr(valid[feature_col], valid[target_col])
        if verbose:
            signal = '✅ KEEP' if abs(ic) > 0.02 else '❌ WEAK'
            print(f"{signal} | {feature_col:30s} | IC={ic:+.4f} | p={pval:.4f} | n={len(valid):,}")
        return ic, pval

# test_feature_ic(df, 'feat_1', 'target', group_col='date')
print(f"{elapsed()} IC tester ready")


---
## 🤖 Phase 6: Model + Validation `[2:10–2:50]`

**Cardinal rules for financial ML:**
1. **Never shuffle** time series data in CV
2. Use **TimeSeriesSplit** or **purged CV** (with gap)
3. Report **OOS metrics only** — in-sample is meaningless
4. Prefer **simple models first** — if LightGBM >> Ridge, investigate why

**Target metrics:**
- IC / Spearman rank correlation (directional accuracy)
- ICIR = IC / std(IC) (consistency — more important than level)
- Hit rate = % of periods with positive IC
- Sharpe of paper portfolio based on signal


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

def run_timeseries_cv(df, features, target_col, date_col,
                      model_type='ridge', n_splits=5, gap=0):
    """
    Time-series cross-validation with optional purge gap.
    gap: number of periods to drop between train and test (purge).
    """
    df = df.sort_values(date_col).dropna(subset=features+[target_col]).copy()
    X  = df[features].values
    y  = df[target_col].values
    dates = df[date_col].values

    if model_type == 'ridge':
        model = Pipeline([('sc', RobustScaler()), ('reg', Ridge(alpha=1.0))])
    elif model_type == 'lasso':
        model = Pipeline([('sc', RobustScaler()), ('reg', Lasso(alpha=0.01))])
    elif model_type == 'gbm':
        model = GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                          learning_rate=0.05, subsample=0.8, random_state=42)
    elif model_type == 'rf':
        model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    tscv   = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    fold_ics = []; fold_r2s = []; oos_preds = np.full(len(y), np.nan)

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        oos_preds[te_idx] = y_pred

        ic, _   = stats.spearmanr(y_pred, y_te)
        r2      = r2_score(y_te, y_pred)
        fold_ics.append(ic); fold_r2s.append(r2)
        print(f"  Fold {fold+1}: IC={ic:+.4f}  R²={r2:.4f}  n_test={len(te_idx):,}")

    mean_ic = np.mean(fold_ics)
    icir    = mean_ic / (np.std(fold_ics) + 1e-9)
    print(f"\n{'='*45}")
    print(f"Model: {model_type.upper()}")
    print(f"Mean IC:   {mean_ic:+.4f}")
    print(f"ICIR:      {icir:+.4f}")
    print(f"Hit rate:  {np.mean(np.array(fold_ics)>0):.0%} of folds positive IC")
    print(f"Mean R²:   {np.mean(fold_r2s):.4f}")
    print(f"{'='*45}")
    return {'mean_ic': mean_ic, 'icir': icir, 'fold_ics': fold_ics, 'oos_preds': oos_preds}

# features = ['feat_1', 'feat_2', 'feat_3']
# results = run_timeseries_cv(df, features, target_col='target', date_col='date',
#                              model_type='ridge', n_splits=5, gap=1)
print(f"{elapsed()} CV framework ready — options: 'ridge', 'lasso', 'gbm', 'rf'")


In [ ]:
# ── Multiple testing correction ───────────────────────────────────────────
# When testing many features, correct for false discovery rate (BHY)

def bhy_correction(pvalues, features, alpha=0.05):
    """
    Benjamini-Hochberg-Yekutieli correction for multiple testing.
    Use after testing many features to avoid p-hacking.
    Returns features that survive correction.
    """
    n = len(pvalues)
    sorted_idx  = np.argsort(pvalues)
    sorted_pvals= np.array(pvalues)[sorted_idx]
    sorted_feats= np.array(features)[sorted_idx]

    # BHY threshold: p(k) <= k/n * alpha / sum(1/i for i in 1..n)
    c_n = sum(1/i for i in range(1, n+1))
    bhy_threshold = np.array([(k+1)/n * alpha / c_n for k in range(n)])
    significant = sorted_pvals <= bhy_threshold

    survivors = sorted_feats[significant]
    print(f"BHY correction (alpha={alpha}): {len(survivors)}/{n} features survive")
    print(f"Survivors: {list(survivors)}")
    return list(survivors)

# pvals = [0.001, 0.03, 0.08, 0.15, 0.42]
# feats = ['feat_1','feat_2','feat_3','feat_4','feat_5']
# bhy_correction(pvals, feats)
print(f"{elapsed()} Multiple testing correction ready")


---
## 📋 Phase 7: Findings Summary `[2:50–3:20]`

**This is what you present to the SQ team. Write it clearly.**

Structure:
1. **What did the data tell you?** (2–3 key observations)
2. **What signals did you find?** (IC, ICIR, statistical significance)
3. **What would you do with 2 more hours?**
4. **What are the risks / limitations of your findings?**

---


In [ ]:
# ── Session timer & summary template ─────────────────────────────────────
print(f"{'='*55}")
print(f"SESSION COMPLETE: {elapsed()}")
print(f"{'='*55}")

FINDINGS = {
    'dataset':       '',   # Dataset name
    'target':        '',   # What were you predicting?
    'n_rows':        '',   # Sample size
    'date_range':    '',   # Time period

    # What you found
    'key_observation_1': '',
    'key_observation_2': '',
    'key_observation_3': '',

    # Best signal
    'best_feature':  '',
    'best_ic':       '',   # e.g. '+0.043'
    'best_icir':     '',   # e.g. '1.8'
    'model_type':    '',   # ridge / gbm / etc
    'oos_r2':        '',

    # Next steps
    'if_2_more_hours': '',
    'data_requested':  '',   # What additional data would help?
    'main_risk':       '',   # Biggest concern about the findings
}

print("FINDINGS SUMMARY:")
for k,v in FINDINGS.items():
    if v: print(f"  {k:25s}: {v}")
    else: print(f"  {k:25s}: [NOT FILLED]")


---
## 🎯 Post-Session Review

After each practice session, answer these 5 questions honestly:

**1. Did I narrate my reasoning out loud?**  
*(On the actual DSI you need to explain every decision verbally)*

**2. Where did I get stuck and why?**  
*(Identify the weakness to fix in the next session)*

**3. Did I check for leakage before modelling?**  
*(This is the most common mistake under pressure)*

**4. Did I write hypotheses before building features?**  
*(Data mining without hypotheses is not research)*

**5. Could a non-technical PM understand my summary?**  
*(If not, your communication needs work)*

---

*Template by Sergey Nefedov | Squarepoint DSI Preparation | github.com/Sergpreneur*
